# Training dataset — build metadata CSV

Companion to `del/SVI_Val.ipynb`, which evaluates the **validation split** of the YOLO waste detector:

1. Read image paths from `dataSet/val.txt`
2. Map each image to its YOLO label in `labels/`
3. Compare ground-truth labels with model predictions (TP / FP / FN / TN)
4. Optionally run CPM on YOLO positives and export confusion matrices

This notebook starts the same way for the **training split**, but instead of model evaluation we build a tabular inventory like `Correct_SVI.csv` / the screenshot you shared:

| img_name | year | month | day | hour | lat | lon | panoid | img_dir | exist | collection | class | yolo_bbox | yolo_num |

Metadata comes from the existing `*_basic.csv` lookup tables (`Google_basic.csv`, `Faith_basic.csv`, `ZWL_basic.csv`). Bounding boxes are read from the training labels and converted to pixel coordinates (same list-of-lists format as `Correct_SVI.csv`). `yolo_conf` is omitted for now.

`collection` is the data source: **GSVI** if `img_name` ends with `_0`, `_90`, `_180`, or `_270` (Google Street View headings); otherwise **SC** (self-collected). This should align with `img_dir` (`Google/...` → GSVI; `Faith/` or `ZWL/` → SC).

`class` records each image's role in YOLO training: `waste` if the label file contains at least one bounding box, otherwise `background`.

In [50]:
import ast
import json
import os
import re
from pathlib import Path

import pandas as pd
from PIL import Image

In [51]:
DATA_ROOT = Path("/Users/wenlanzhang/Downloads/PhD_UCL/Data")
IMG_ROOT = DATA_ROOT / "Waste/img"
TRAIN_ROOT = IMG_ROOT / "Train0315_695"
OUTPUT_DIR = Path("/Users/wenlanzhang/PycharmProjects/Chapter_waste/6_sensitivity")
OUTPUT_CSV = OUTPUT_DIR / "Train0315_695_training.csv"

SPLIT_FILE = TRAIN_ROOT / "dataSet/trainval.txt"  # all 696 labelled images; use train.txt for train-only
IMAGES_DIR = TRAIN_ROOT / "images"
LABELS_DIR = TRAIN_ROOT / "labels"

BASIC_CSVS = [
    IMG_ROOT / "Google_basic.csv",
    IMG_ROOT / "Faith_basic.csv",
    IMG_ROOT / "ZWL_basic.csv",
]

## 1. Load the training image list

`trainval.txt` lists every image used in the YOLO run (626 train + 70 val). Paths inside the file point at the older `GoogleStreetView/Train/Train0315/images/` tree, but the same basenames live locally under `Train0315_695/images/`.

In [52]:
with open(SPLIT_FILE) as f:
    split_paths = [line.strip() for line in f if line.strip()]

img_names = [Path(p).stem for p in split_paths]
print(f"Images in split file: {len(img_names)}")
print("Examples:", img_names[:5])

Images in split file: 696
Examples: ['LkN9dmH-f_QJFFxGOk7PEA_90', 'VSNF3NE5eykDgUUAqoayFg_270', '459eb9b0-UPoDfWE8S-Ya6uSXDf3w8Q_0', '701ab709-UrRoUBKATo0lZ4iSFwxUng_0', 'f10f8795-eIZ7kN3HHMhLCIFz3M5mNw_0']


## 2. Load metadata lookup tables

These are the same source tables used when building `Correct_SVI.csv` and `Combined_SVI.csv`.

In [53]:
meta_parts = []
for path in BASIC_CSVS:
    part = pd.read_csv(path)
    part["meta_source"] = path.stem.replace("_basic", "")
    meta_parts.append(part)

meta = pd.concat(meta_parts, ignore_index=True)
meta_by_name = meta.set_index("img_name", drop=False)
meta_by_panoid = meta.dropna(subset=["panoid"]).drop_duplicates("panoid", keep="first").set_index("panoid", drop=False)

print(meta["meta_source"].value_counts())
meta.head(3)

meta_source
Google    453364
ZWL         2930
Faith        817
Name: count, dtype: int64


,panoid,lat,lon,year,month,img_name,img_dir,exist,day,hour,meta_source
0,khlXWXsuIX7rodKwi1fDAw,-1.347984,36.904885,2021,10,khlXWXsuIX7rodKwi1fDAw_0,Google/k/h/,True,NaN,NaN,Google
1,khlXWXsuIX7rodKwi1fDAw,-1.347984,36.904885,2021,10,khlXWXsuIX7rodKwi1fDAw_90,Google/k/h/,True,NaN,NaN,Google
2,khlXWXsuIX7rodKwi1fDAw,-1.347984,36.904885,2021,10,khlXWXsuIX7rodKwi1fDAw_180,Google/k/h/,True,NaN,NaN,Google


## 3. Helper functions

In [54]:
UUID_PREFIX = re.compile(r"^[0-9a-f]{8}-", re.I)
GSVI_SUFFIX = re.compile(r"_(0|90|180|270)$")
JUNK_NAMES = {".DS_Store"}


def collection_from_name(img_name: str) -> str:
    """GSVI if the filename ends with a Street View heading; otherwise self-collected."""
    return "GSVI" if GSVI_SUFFIX.search(img_name) else "SC"


def collection_from_img_dir(img_dir) -> str | None:
    if pd.isna(img_dir):
        return None
    path = str(img_dir)
    if path.startswith("Google/"):
        return "GSVI"
    if path.startswith("Faith/") or path.startswith("ZWL/"):
        return "SC"
    return None


def candidate_names(img_name: str) -> list[str]:
    """Try direct name, then strip an 8-char hex prefix used on some self-collected files."""
    names = [img_name]
    if UUID_PREFIX.match(img_name):
        names.append(img_name.split("-", 1)[1])
    return list(dict.fromkeys(names))


def panoid_from_name(img_name: str) -> str:
    base = img_name.split("-", 1)[1] if UUID_PREFIX.match(img_name) else img_name
    if "_" in base:
        return base.rsplit("_", 1)[0]
    return base


def lookup_metadata(img_name: str) -> dict | None:
    for cand in candidate_names(img_name):
        if cand in meta_by_name.index:
            row = meta_by_name.loc[cand]
            return row.iloc[0].to_dict() if isinstance(row, pd.DataFrame) else row.to_dict()

    panoid = panoid_from_name(img_name)
    if panoid in meta_by_panoid.index:
        return meta_by_panoid.loc[panoid].to_dict()
    return None


def yolo_to_pixel_bbox(line: str, width: int, height: int) -> list[float]:
    """Convert one YOLO label line (class cx cy w h, normalised) to [x1, y1, x2, y2] pixels."""
    parts = line.strip().split()
    _, cx, cy, bw, bh = map(float, parts[:5])
    x1 = (cx - bw / 2) * width
    y1 = (cy - bh / 2) * height
    x2 = (cx + bw / 2) * width
    y2 = (cy + bh / 2) * height
    return [x1, y1, x2, y2]


def read_label_bboxes(img_name: str) -> tuple[list[list[float]], int]:
    label_path = LABELS_DIR / f"{img_name}.txt"
    image_path = IMAGES_DIR / f"{img_name}.jpg"

    if not label_path.exists() or label_path.stat().st_size == 0:
        return [], 0

    if not image_path.exists():
        return [], 0

    with Image.open(image_path) as im:
        width, height = im.size

    boxes = []
    with open(label_path) as f:
        for line in f:
            line = line.strip()
            if line:
                boxes.append(yolo_to_pixel_bbox(line, width, height))
    return boxes, len(boxes)

## 4. Build the training table

In [55]:
rows = []
missing_meta = []
missing_image = []

for img_name in img_names:
    if img_name in JUNK_NAMES:
        continue

    image_path = IMAGES_DIR / f"{img_name}.jpg"
    if not image_path.exists():
        missing_image.append(img_name)

    meta_row = lookup_metadata(img_name)
    if meta_row is None:
        missing_meta.append(img_name)
        meta_row = {
            "year": pd.NA,
            "month": pd.NA,
            # "day": pd.NA,
            # "hour": pd.NA,
            "lat": pd.NA,
            "lon": pd.NA,
            "panoid": pd.NA,
            "img_dir": pd.NA,
            "exist": False,
        }

    yolo_bbox, yolo_num = read_label_bboxes(img_name)
    training_class = "waste" if yolo_num > 0 else "background"
    collection = collection_from_name(img_name)

    rows.append(
        {
            "img_name": img_name,
            "year": meta_row.get("year"),
            "month": meta_row.get("month"),
            # "day": meta_row.get("day"),
            # "hour": meta_row.get("hour"),
            "lat": meta_row.get("lat"),
            "lon": meta_row.get("lon"),
            "panoid": meta_row.get("panoid"),
            "img_dir": meta_row.get("img_dir"),
            "exist": bool(image_path.exists()) if image_path.exists() else meta_row.get("exist", False),
            "collection": collection,
            "class": training_class,
            # "yolo_bbox": json.dumps(yolo_bbox),
            # "yolo_num": yolo_num,
        }
    )

train_df = pd.DataFrame(rows)
print(f"Rows built: {len(train_df)}")
print("\nCollection:")
print(train_df["collection"].value_counts().to_string())
print("\nClass:")
print(train_df["class"].value_counts().to_string())
print(f"Missing metadata: {len(missing_meta)}")
print(f"Missing local image file: {len(missing_image)}")
if missing_meta:
    print("Unmatched img_name examples:", missing_meta[:10])
train_df.head()

Rows built: 695

Collection:
collection
GSVI    610
SC       85

Class:
class
waste         353
background    342
Missing metadata: 8
Missing local image file: 0
Unmatched img_name examples: ['1f7e5b1e-3767618616712176', '12fe849c-WechatIMG1048', 'b177368d-WechatIMG1047', 'e46a1e51-WechatIMG1046', '44c2da02-IMG_6653', 'aa90b6e2-WechatIMG1049', 'dddd727b-IMG_6652', '37eb2376-310211603998748']


,img_name,year,month,lat,lon,panoid,img_dir,exist,collection,class
0,LkN9dmH-f_QJFFxGOk7PEA_90,2018,10,-1.374241,36.779327,LkN9dmH-f_QJFFxGOk7PEA,Google/L/k/,True,GSVI,background
1,VSNF3NE5eykDgUUAqoayFg_270,2021,7,-1.238584,36.851001,VSNF3NE5eykDgUUAqoayFg,Google/V/S/,True,GSVI,background
2,459eb9b0-UPoDfWE8S-Ya6uSXDf3w8Q_0,2018,4,-1.320441,36.892761,UPoDfWE8S-Ya6uSXDf3w8Q,Google/U/P/,True,GSVI,waste
3,701ab709-UrRoUBKATo0lZ4iSFwxUng_0,2021,8,-1.312264,36.857056,UrRoUBKATo0lZ4iSFwxUng,Google/U/r/,True,GSVI,waste
4,f10f8795-eIZ7kN3HHMhLCIFz3M5mNw_0,2021,9,-1.186151,36.905804,eIZ7kN3HHMhLCIFz3M5mNw,Google/e/I/,True,GSVI,waste


## 5. Quick checks

In [56]:
from IPython.display import Markdown, display

n = len(train_df)

# Marginal counts
collection_tbl = (
    train_df["collection"]
    .value_counts()
    .reindex(["GSVI", "SC"])
    .rename("Count")
    .to_frame()
    .assign(**{"% of total": lambda d: (d["Count"] / n * 100).round(1)})
)
collection_tbl["Count"] = collection_tbl["Count"].map("{:,}".format)

class_tbl = (
    train_df["class"]
    .value_counts()
    .reindex(["waste", "background"])
    .rename("Count")
    .to_frame()
    .assign(**{"% of total": lambda d: (d["Count"] / n * 100).round(1)})
)
class_tbl["Count"] = class_tbl["Count"].map("{:,}".format)

# Stratified table: counts + within-collection %
counts = pd.crosstab(train_df["collection"], train_df["class"]).reindex(index=["GSVI", "SC"], columns=["background", "waste"])
within_pct = counts.div(counts.sum(axis=1), axis=0).mul(100).round(1)

stratified_tbl = counts.copy()
stratified_tbl["Total"] = counts.sum(axis=1)
stratified_tbl["Waste %"] = within_pct["waste"]
stratified_tbl.loc["Total"] = [
    counts["background"].sum(),
    counts["waste"].sum(),
    n,
    (train_df["class"] == "waste").mean() * 100,
]

display(Markdown(f"### Training set summary — **{n:,}** images"))
display(Markdown("**By collection**"))
display(collection_tbl)
display(Markdown("**By YOLO class**"))
display(class_tbl)
display(Markdown("**Stratified: collection × class**"))
display(
    stratified_tbl.rename_axis("Collection")
    .style.format({
        "background": "{:,.0f}",
        "waste": "{:,.0f}",
        "Total": "{:,.0f}",
        "Waste %": "{:.1f}%",
    }, na_rep="")
    .set_caption("Counts with waste share within each collection")
)

### Training set summary — **695** images

**By collection**

,Count,% of total
collection,,
GSVI,610,87.8
SC,85,12.2


**By YOLO class**

,Count,% of total
class,,
waste,353,50.8
background,342,49.2


**Stratified: collection × class**

class,background,waste,Total,Waste %
Collection,,,,
GSVI,295,315,610,51.6%
SC,47,38,85,44.7%
Total,342,353,695,50.8%


In [57]:
from IPython.display import Markdown, display

train_df["collection_from_dir"] = train_df["img_dir"].map(collection_from_img_dir)
known = train_df["collection_from_dir"].notna()
mismatches = train_df.loc[known & (train_df["collection"] != train_df["collection_from_dir"])]
missing_dir = train_df[train_df["collection_from_dir"].isna()]

# QA headline
qa_tbl = pd.DataFrame(
    {
        "Check": [
            "Rows with known img_dir",
            "Name rule matches img_dir",
            "Mismatches",
            "Missing img_dir",
        ],
        "Result": [
            f"{known.sum():,} / {len(train_df):,}",
            f"{known.sum() - len(mismatches):,} / {known.sum():,}",
            f"{len(mismatches):,}",
            f"{len(missing_dir):,}",
        ],
        "Status": [
            "OK" if known.sum() > 0 else "—",
            "OK" if len(mismatches) == 0 else "Review",
            "OK" if len(mismatches) == 0 else "Review",
            "Note" if len(missing_dir) > 0 else "OK",
        ],
    }
)

# collection (name) vs collection (img_dir)
check_df = train_df.assign(
    collection_from_dir=train_df["collection_from_dir"].fillna("missing img_dir")
)
cross_tbl = (
    pd.crosstab(check_df["collection"], check_df["collection_from_dir"])
    .reindex(index=["GSVI", "SC"], fill_value=0)
)

# Top img_dir roots
img_dir_tbl = (
    train_df["img_dir"]
    .fillna("<missing>")
    .map(lambda x: "Google/…" if str(x).startswith("Google/") else x)
    .value_counts()
    .head(10)
    .rename("Count")
    .to_frame()
    .assign(**{"% of total": lambda d: (d["Count"] / len(train_df) * 100).round(1)})
)

display(Markdown("### Collection consistency check"))
display(Markdown("_Does the filename rule (`_0/_90/_180/_270` → GSVI) agree with `img_dir`?_"))
display(qa_tbl.style.hide(axis="index"))

display(Markdown("**Name rule vs `img_dir`**"))
display(
    cross_tbl.rename_axis("From img_name")
    .rename_axis("From img_dir", axis=1)
    .style.format("{:,.0f}")
    .set_caption("Cross-tab: collection from filename vs collection inferred from img_dir")
)

if len(missing_dir):
    display(Markdown(f"**Missing `img_dir`** ({len(missing_dir)} rows)"))
    display(
        missing_dir[["img_name", "collection"]]
        .reset_index(drop=True)
        .style.hide(axis="index")
    )

display(Markdown("**Top `img_dir` groups**"))
display(img_dir_tbl.style.format({"Count": "{:,.0f}", "% of total": "{:.1f}%"}))

### Collection consistency check

_Does the filename rule (`_0/_90/_180/_270` → GSVI) agree with `img_dir`?_

Check,Result,Status
Rows with known img_dir,687 / 695,OK
Name rule matches img_dir,687 / 687,OK
Mismatches,0,OK
Missing img_dir,8,Note


**Name rule vs `img_dir`**

From img_dir,GSVI,SC,missing img_dir
From img_name,,,
GSVI,610,0,0
SC,0,77,8


**Missing `img_dir`** (8 rows)

img_name,collection
1f7e5b1e-3767618616712176,SC
12fe849c-WechatIMG1048,SC
b177368d-WechatIMG1047,SC
e46a1e51-WechatIMG1046,SC
44c2da02-IMG_6653,SC
aa90b6e2-WechatIMG1049,SC
dddd727b-IMG_6652,SC
37eb2376-310211603998748,SC


**Top `img_dir` groups**

,Count,% of total
img_dir,,
Google/…,610,87.8%
ZWL/,77,11.1%
,8,1.2%


## 6. Export CSV

In [58]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_df = train_df.drop(columns=["collection_from_dir"], errors="ignore")
export_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(export_df):,} rows to {OUTPUT_CSV}")

Saved 695 rows to /Users/wenlanzhang/PycharmProjects/Chapter_waste/6_sensitivity/Train0315_695_training.csv


## Next steps

- Compare class balance between `train.txt` and `val.txt` (the validation notebook workflow).
- Join this table back to `Correct_SVI.csv` / full SVI inventories to see what fraction of detected waste sits in the training set.
- Add `yolo_conf` later if we re-run inference on training images.
- Flag or drop the few images with no metadata match (mostly prefixed `WechatIMG*` / `IMG_*` filenames).